In [1]:
import pandas as pd
import numpy as np
import gc

### Train / Test 데이터 병합

- train

In [ ]:
# train data
customer_train_df = pd.read_parquet("../data/Dacon/customer_train_cleaned.parquet")
credit_train_df = pd.read_parquet("../data/Dacon/credit_train_cleaned.parquet")
sales_train_df = pd.read_parquet("../data/Dacon/sales_train_cleaned.parquet")
billing_train_df = pd.read_parquet("../data/Dacon/billing_train_cleaned.parquet")
balance_train_df = pd.read_parquet("../data/Dacon/balance_train_cleaned.parquet")
channel_train_df = pd.read_parquet("../data/Dacon/channel_train_cleaned.parquet")
marketing_train_df = pd.read_parquet("../data/Dacon/marketing_train_cleaned.parquet")
performance_train_df = pd.read_parquet("../data/Dacon/performance_train_cleaned.parquet")

In [ ]:
train_df = customer_train_df.merge(credit_train_df, on=['기준년월', 'ID'], how='left')
train_df = train_df.merge(sales_train_df, on=['기준년월', 'ID'], how='left')
train_df = train_df.merge(billing_train_df, on=['기준년월', 'ID'], how='left')
train_df = train_df.merge(balance_train_df, on=['기준년월', 'ID'], how='left')
train_df = train_df.merge(channel_train_df, on=['기준년월', 'ID'], how='left')
train_df = train_df.merge(marketing_train_df, on=['기준년월', 'ID'], how='left')
train_df = train_df.merge(performance_train_df, on=['기준년월', 'ID'], how='left')

In [5]:
for col in train_df.select_dtypes(include='int64').columns:
    if train_df[col].max() < 2_147_483_647:
        train_df[col] = train_df[col].astype('int32') # 메모리 줄이기 위해 int64 ->int32

In [7]:
train_df.to_parquet('../data/Data/train_df_cleaned.parquet')

- test

In [3]:
# test data
customer_test_df = pd.read_parquet("../data/Dacon/customer_test_cleaned.parquet")
credit_test_df = pd.read_parquet("../data/Dacon/credit_test_cleaned.parquet")
sales_test_df = pd.read_parquet("../data/Dacon/sales_test_cleaned.parquet")
billing_test_df = pd.read_parquet("../data/Dacon/billing_test_cleaned.parquet")
balance_test_df = pd.read_parquet("../data/Dacon/balance_test_cleaned.parquet")
channel_test_df = pd.read_parquet("../data/Dacon/channel_test_cleaned.parquet")
marketing_test_df = pd.read_parquet("../data/Dacon/marketing_test_cleaned.parquet")
performance_test_df = pd.read_parquet("../data/Dacon/performance_test_cleaned.parquet")

In [4]:
test_df = customer_test_df.merge(credit_test_df, on=['기준년월', 'ID'], how='left')
test_df = test_df.merge(sales_test_df, on=['기준년월', 'ID'], how='left')
test_df = test_df.merge(billing_test_df, on=['기준년월', 'ID'], how='left')
test_df = test_df.merge(balance_test_df, on=['기준년월', 'ID'], how='left')
test_df = test_df.merge(channel_test_df, on=['기준년월', 'ID'], how='left')
test_df = test_df.merge(marketing_test_df, on=['기준년월', 'ID'], how='left')
test_df = test_df.merge(performance_test_df, on=['기준년월', 'ID'], how='left')

for col in test_df.select_dtypes(include='int64').columns:
    if test_df[col].max() < 2_147_483_647:
        test_df[col] = test_df[col].astype('int32')

for col in test_df.select_dtypes(include='float64').columns:
    test_df[col] = test_df[col].astype('float32')

In [5]:
test_df.to_parquet('../data/Data/test_df_cleaned.parquet')

### Modeling(1) - feature importance

In [6]:
import sklearn
from sklearn.utils.class_weight import compute_class_weight
import imblearn
from imblearn.over_sampling import SMOTE
import xgboost as xgb
from xgboost import XGBClassifier

In [9]:
train_df = pd.read_parquet('../data/Data/train_df_cleaned.parquet')

feature_cols = [col for col in train_df.columns if col not in ["ID", "Segment"]]
X = train_df[feature_cols].copy()
y = train_df["Segment"].copy()
y = y.map({'A':0, 'B':1,'C':2,'D':3,'E':4})

del train_df
gc.collect()

# 클래스 weight 계산
classes = np.unique(y)
weights = compute_class_weight(class_weight='balanced', classes=classes, y=y)
class_weights = dict(zip(classes, weights))

# 각 샘플에 대해 weight 매핑
w_train = pd.Series(y).map(class_weights)

# 전체 feature로 XGBoost 학습 (변수 중요도 추출용)
temp_model = xgb.XGBClassifier(
    objective='multi:softprob',
    num_class=5,
    eval_metric='mlogloss',
    n_estimators=700,
    tree_method='hist',
    device='cuda',
    random_state=42
    )

temp_model.fit(X, y, sample_weight = w_train, verbose=False)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device='cuda', early_stopping_rounds=None,
              enable_categorical=False, eval_metric='mlogloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=None, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=None, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=700, n_jobs=None, num_class=5, ...)

In [10]:
# XGBoost 기준 중요도 상위 300개 변수 추출
importance_df = pd.DataFrame({
    'feature': X.columns,
    'importance': temp_model.feature_importances_
}).sort_values(by='importance', ascending=False)

top300_features = importance_df.head(300)['feature'].tolist()

print(top300_features)

top300_df = pd.DataFrame({'feature': top300_features})
top300_df.to_csv(
    "../data/Dacon/top300_features_XGB_balanced.csv",
    index=False,
    encoding="utf-8-sig"
)

['카드이용한도금액_A수준복합', '이용금액_선결제_B0M', '할부금액_무이자_3M_R12M', '정상청구원금_B5M', '이용금액_신용_R12M', '평잔_CA_해외_3M', '이용금액_체크_B0M', '정상청구원금_B0M', '이용월평균이용금액_체크_R3M', '카드이용한도금액_B1M', '잔액_리볼빙CA이월_B0M', '할인금액_청구서_B0M', '총납부금_B5M', 'CA한도금액', '이용월평균이용금액_체크_R12M', '카드이용한도금액_B2M', '이용개월수_카드론_R6M', '할부금액총합_무이자_R12M', 'RP건수_학습비_B0M', '총납부금_B2M', '방문일수_앱_B0M', '탈회횟수_발급1년이내', '연체입금원금_B5M', '평잔_CA_해외_6M', '증감_RP건수_전기_전월', '이용월평균이용금액_할부_유이자_R6M', '건수별평균이용금액_CA_R6M', '이용금액_할부_무이자_B0M', '이용금액_체크_R12M', '월중평잔_할부_B0M', 'RP건수_아파트_B0M', 'rv최초시작후경과일', '이용개월수_체크_R3M', 'RP후경과월_아파트', '최대이용금액_체크_R12M', '이용개월수_할부_무이자_R12M', '정상청구원금_B2M', '건수별평균이용금액_체크_R3M', '건수별평균이용금액_신용_R12M', '포인트_마일리지_건별_R3M', '이용여부_CA', '이용금액_해외', '이용금액_오프라인_B0M', '정시납부금_B2M', '이용금액_신용_B0M', '여유_총이용금액', '입회경과개월수_신용', '연속유실적개월수_기본_24M_카드', '월중평잔_CA_B0M', '총납부금_B0M', '최종카드론_대출이율', '카드이용한도금액', '포인트_마일리지_건별_B0M', '할인금액_청구서_R3M', '이용금액_신판_R12M', '이용개월수_결제일_R6M', '이용월평균이용금액_신용_R12M', '이용건수_카드론_R12M', '건수별평균이용금액_체크_R6M', 'CA이자율_할인전', '이용금액_할부_R12M', '이용금액_체크_R6M'

### Modeling(2) - final model train

#### XGBoost 모델 학습

In [12]:
train_df = pd.read_parquet('../data/Data/train_df_cleaned.parquet')

feature_cols = [col for col in train_df.columns if col not in ["ID", "Segment"]]
X = train_df[feature_cols].copy()
y = train_df["Segment"].copy()
y = y.map({'A':0, 'B':1,'C':2,'D':3,'E':4})
inverse_label_map = {0: 'A', 1: 'B', 2: 'C', 3: 'D', 4: 'E'}

# 변수 300개 사용
top300_df = pd.read_csv("../data/Dacon/top300_features_XGB_balanced.csv")
top300_features = top300_df['feature'].tolist()
X_top300 = X[top300_features]

# 오버샘플링
smote = SMOTE(sampling_strategy={0: 30000, 1: 30000, 2: 250000}, random_state=42)
X_resampled, y_resampled = smote.fit_resample(X_top300, y)

# 클래스별 weight 계산
classes = np.unique(y_resampled)
weights = compute_class_weight(class_weight='balanced', classes=classes, y=y_resampled)
class_weights = dict(zip(classes, weights))
sample_weights = pd.Series(y_resampled).map(class_weights)

for cls in sorted(class_weights):
    print(f"클래스 {cls}: weight = {class_weights[cls]:.2f}")

클래스 0: weight = 17.21
클래스 1: weight = 17.21
클래스 2: weight = 2.07
클래스 3: weight = 1.48
클래스 4: weight = 0.27


In [13]:
xgb_model = xgb.XGBClassifier(
    objective='multi:softprob',
    num_class=5,
    eval_metric='mlogloss',
    n_estimators=5000,
    tree_method='hist',
    device='cuda',
    random_state=42
    )

# 모델 학습 (검증 없이 전체 데이터 사용)
xgb_model.fit(
    X_resampled, y_resampled,
    sample_weight=sample_weights,
    verbose=False
)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device='cuda', early_stopping_rounds=None,
              enable_categorical=False, eval_metric='mlogloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=None, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=None, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=5000, n_jobs=None, num_class=5, ...)

In [14]:
xgb_model.save_model('../data/Dacon/softvoting_xgb_xgb변수_3_3_25.json')

### Predict

In [ ]:
test_df = pd.read_parquet("../data/Data/test_df_cleaned.parquet")

# 변수 300개 사용
top300_df = pd.read_csv("../data/Dacon/top300_features_XGB_balanced.csv")
top300_features = top300_df['feature'].tolist()

# 학습한 모델들 불러오기
xgb_model_loaded = xgb.XGBClassifier()
xgb_model_loaded.load_model('../data/Dacon/softvoting_xgb_xgb변수_3_3_25.json')

In [8]:
# test 데이터 준비
X_test = test_df[top300_features]

In [ ]:
# XGB predict_proba
proba_xgb = xgb_model_loaded.predict_proba(X_test, iteration_range=(0, 5000))

# 단일 모델 결과 그대로 사용
ensemble_proba = proba_xgb
ensemble_preds = np.argmax(ensemble_proba, axis=1)

inverse_label_map = {0: 'A', 1: 'B', 2: 'C', 3: 'D', 4: 'E'}
ensemble_preds_label = pd.Series(ensemble_preds).map(inverse_label_map)

# 예측값 test_df에 추가
test_df["pred_label"] = ensemble_preds_label

# ID별 mode aggregation
submission = test_df.groupby("ID")["pred_label"] \
    .agg(lambda x: x.value_counts().idxmax()) \
    .reset_index()

submission.columns = ["ID", "Segment"]

# 저장
submission.to_csv(
    "../data/Dacon/submission_final.csv",
    index=False,
    encoding="utf-8-sig"
)

print("결과 저장 완료")

결과 저장 완료!
